# Wordle game engine

A `Wordle` class that holds a secret word and scores guesses against it. Results are printed in the solver's input notation: `!` after a green letter, `?` after a yellow one, and nothing after a gray one.

Repeated letters are scored in two passes. Greens are marked first and use up their letter. Yellows are then handed out from the letters that are left, from left to right.


In [1]:
import pandas as pd, numpy as np, os,glob

In [2]:
import string

In [3]:
all_letters = list(string.ascii_lowercase)

In [4]:
# def score_guess(guess,wordle):
#     """ """
#     #maybe don't get wordle letter counts each guess or it's done more often than necessary - if it was a class then itd be
#     letters, counts = np.unique(list(wordle),return_counts=True)
#     wordle_letter_counts = pd.Series(counts,index=letters)
#     results = dict()
#     for i,letter in enumerate(guess):
#         results[i] = {'letter':letter,'color':None}
#     # get the green ones first
#     for i,letter in enumerate(guess):
#         if guess[i] == wordle[i]:
#             results[i]['color'] = 'g'
#             #decrement the counts in order to have correct number of colors per letter (in case of multiples)
#             wordle_letter_counts[letter] -= 1

#     for i,letter in enumerate(guess):
#         #letter may not be in wordle_letter_counts
#         #could case wordle_letter_counts.index to set for quick searching but this is small scale
#         if letter in wordle_letter_counts.index:
#             if results[i]['color'] == 'g':
#                 pass
#             elif (wordle_letter_counts[letter] > 0) and (results[i]['color'] != 'g'):
#                 results[i]['color'] = 'y'
#                 wordle_letter_counts[letter] -= 1
#             else:
#                 results[i]['color'] = 'b'
#         else:
#             results[i]['color'] = 'b'

#     print(wordle_letter_counts)
#     return results       

In [5]:
from getpass import getpass

class Wordle():
    """ """ 
    

    def __init__(self,wordle=None):
        """ """
        if wordle is None:
            self.wordle = getpass('Please enter in the wordle\n').lower()
        else:
            self.wordle = wordle.lower()


        ## TODO: add functionality for a random word to be chosen as the wordle
        letters, counts = np.unique(list(self.wordle),return_counts=True)
        self.wordle_letter_counts = pd.Series(counts,index=letters)

        #don't allow inappropriate letters as 
        inappropriate_letters = set(self.wordle.lower()) - set(string.ascii_lowercase)
        if len(inappropriate_letters) > 0:
            raise ValueError('the following characters are not allowed:',inappropriate_letters)

        ## check to see if its a real word?? 

    def _beautify_guess_results(self,guess_results):
        """ """
        # two ways to print output, one string of gby 
        # or follow format where it's got a question mark appended if y, and ! if g, else blank
        output1 = ''
        output2 = ''
        legend = {'b':'','g':'!','y':'?'}
        for i in guess_results.keys():
            #print(i,guess_results[i]['letter'])
            output1 += (guess_results[i]['letter'] + legend[guess_results[i]['color']])
            output2 += (guess_results[i]['color'])

        # output 1 is better 
        return output1
        # return output2

    def score_guess(self,guess):
        """ """        
        if len(guess) != len(self.wordle):
            raise ValueError(f'guess needs to have {len(self.wordle)} letters')
        ## check to see if any letters were submitted that are inappropriate
        inappropriate_letters = set(guess.lower()) - set(string.ascii_lowercase)
        if len(inappropriate_letters) > 0:
            raise ValueError('the following characters are not allowed:',inappropriate_letters)
        #make all the letters lowercase just to avoid simple errors with input
        guess = guess.lower()
        ## need to make a temporary copy of the letter counts otherwise it's modified for subsequent guesses - since Series are mutable
        temporary_wordle_letter_counts = self.wordle_letter_counts.copy()

        results = dict()
        for i,letter in enumerate(guess):
            results[i] = {'letter':letter,'color':None}
        # get the green ones first
        for i,letter in enumerate(guess):
            if guess[i] == self.wordle[i]:
                results[i]['color'] = 'g'
                #decrement the counts in order to have correct number of colors per letter (in case of multiples)
                temporary_wordle_letter_counts[letter] -= 1

        for i,letter in enumerate(guess):
            #letter may not be in  self.wordle_letter_counts
            #could case  self.wordle_letter_counts.index to set for quick searching but this is small scale
            if letter in temporary_wordle_letter_counts.index:
                if results[i]['color'] == 'g':
                    pass
                elif (temporary_wordle_letter_counts[letter] > 0) and (results[i]['color'] != 'g'):
                    results[i]['color'] = 'y'
                    temporary_wordle_letter_counts[letter] -= 1
                else:
                    results[i]['color'] = 'b'
            else:
                results[i]['color'] = 'b'

        print(self._beautify_guess_results(results))
        return results       

    def play_game(self, guesses=None, max_guesses=6):
        """ Play up to max_guesses rounds and return True if the wordle was found.
        Guesses are typed at a prompt, or taken from the `guesses` list to replay a game. """
        if guesses is not None:
            guesses = iter(guesses)
        number_of_guesses = 0
        while number_of_guesses < max_guesses:
            if guesses is None:
                guess = input('Submit a guess - ')
            else:
                guess = next(guesses, None)
                if guess is None:
                    return False # ran out of scripted guesses
                print('Submit a guess -', guess)
            try:
                self.score_guess(guess)
            except ValueError as e:
                ## a typo (wrong length, digits, ...) shouldn't end the game or use up a guess
                print(*e.args)
                continue
            number_of_guesses += 1
            if guess.lower() == self.wordle:
                print(f'you did it in {number_of_guesses}')
                return True
        print(f'out of guesses - the wordle was {self.wordle}')
        return False


In [6]:
## replay of a real game (the wordle was typed in hidden with getpass)
W = Wordle('butch')
W.play_game(guesses=['sauce', 'hunch', 'mulch', 'dutch', 'butch'])

Submit a guess - sauce
sau?c!e
Submit a guess - hunch
hu!nc!h!
Submit a guess - mulch
mu!lc!h!
Submit a guess - dutch
du!t!c!h!
Submit a guess - butch
b!u!t!c!h!
you did it in 5


True

In [7]:
def colors(wordle, guess):
    """ score a guess and return the colors as a string, e.g. 'bbygb' """
    results = Wordle(wordle).score_guess(guess)
    return ''.join(results[i]['color'] for i in results)

def test_score_guess():
    """ """
    assert colors('butch', 'sauce') == 'bbygb'
    assert colors('butch', 'butch') == 'ggggg'
    ## repeated letters: greens use up their letter first, then yellows go left to right
    assert colors('vivid', 'vviiv') == 'gyygb' ## the third v is gray: vivid only has two
    assert colors('vivid', 'ivoiv') == 'yybgy'
    assert colors('eerie', 'leaee') == 'bgbyg'
    assert colors('eerie', 'lease') == 'bgbbg'
    ## bad input
    for wordle, guess in [('vivid', '3acba'), ('vivid', 'ba'), ('cactus', 'blade')]:
        try:
            Wordle(wordle).score_guess(guess)
            raise AssertionError(f'{guess!r} should have been rejected')
        except ValueError:
            pass
    print('all tests passed')

test_score_guess()

sau?c!e
b!u!t!c!h!
v!v?i?i!v
i?v?oi!v?
le!ae?e!
le!ase!
all tests passed


## Play it yourself

Uncomment and run. The first prompt hides the wordle as you type it, so someone else can set the puzzle.


In [8]:
# W = Wordle()
# W.play_game()